<a href="https://colab.research.google.com/github/Poojarautela03/ABTALKS/blob/main/Day29_Building%20AI%20Evaluation%20Systems/Evaluation_Systems_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 29 — Building AI Evaluation Systems
**ABTalks 60-Day AI Challenge · Focus Area: AI Evaluation and Measurement**

You cannot improve what you cannot measure. This notebook builds an **LLM-as-judge**
evaluation framework for the Day 20-style knowledge assistant: a judge that scores every
answer on three precise dimensions, a 20-question hand-labeled dataset, and a regression
test that can block a bad deployment automatically instead of relying on someone noticing
the assistant got worse.

**The three evaluation dimensions, defined precisely:**
- **Groundedness** — the answer is supported *only* by the retrieved context, with no
  fabricated additions.
- **Correctness** — the answer matches the known ground truth.
- **Completeness** — the full question is answered, with nothing important omitted.

**What this notebook does:**
1. Builds `llm_judge(question, context, answer, ground_truth)` scoring all three dimensions 1-5
2. Creates a 20-question hand-labeled dataset for the Day 20-style assistant, 2 per source doc
3. Runs the assistant on all 20 questions and collects every score in a results dictionary
4. Calculates aggregate (average) scores per dimension and identifies the lowest one
5. Builds `regression_test_runner()`, comparing scores against a stored baseline —
   demonstrated against both an unchanged assistant (PASS) and a deliberately regressed one
   (FAIL, on all three dimensions, with the exact drop reported)
6. Documents the honest limitations of LLM-as-judge evaluation, grounded in bugs this
   notebook's own judge actually had while being built

**Note on the judge itself:** as with every previous day, `llm_judge()` runs on a
deterministic offline heuristic scorer by default (see `judge.py`) rather than a real
GPT-4o-mini call, so the notebook is reproducible without an API key. Set `OPENAI_API_KEY`
to swap in a real structured-output judge call — nothing else in the notebook (the dataset,
the runner, the regression logic) needs to change, since they only depend on `llm_judge()`'s
return shape, not its internals.


## 1. The assistant under evaluation

A Day 20-style knowledge assistant: bag-of-words retrieval over a 10-document knowledge base,
plus a generation step. The 20 scripted answers below are **not** all correct — 6 have a
deliberate, realistic flaw (2 wrong facts, 2 that drop part of a multi-part answer, 2 that add
a fabricated detail the context never said), because the whole point of building an evaluation
framework is to find real problems, and a mock assistant that never makes mistakes wouldn't
prove the judge actually catches anything. Which 6 are flawed is not labeled in the code —
that's what the evaluation in section 4 has to find.


In [1]:
%%writefile knowledge_base.py
"""
knowledge_base.py
------------------
The Day 20-style knowledge assistant's corpus, extended to 10 documents so
there's enough ground to write a genuine 20-question evaluation set against.
"""

from typing import Dict, List

KNOWLEDGE_BASE: List[Dict[str, str]] = [
    {"source": "doc_python", "text": "Python was first released by Guido van Rossum in 1991."},
    {"source": "doc_eiffel_tower", "text": "The Eiffel Tower is 330 meters tall and was completed in 1889."},
    {"source": "doc_everest", "text": "Mount Everest is 8849 meters tall, the highest mountain on Earth."},
    {"source": "doc_amazon", "text": "The Amazon rainforest covers approximately 5500000 square kilometers."},
    {"source": "doc_moon_landing", "text": "The Apollo 11 moon landing occurred on July 20, 1969, with Neil Armstrong as the first person to walk on the Moon."},
    {"source": "doc_recursion", "text": "Recursion is a programming technique where a function calls itself to solve smaller instances of a problem. Every recursive function needs a base case to stop the recursive calls, or it will cause a stack overflow."},
    {"source": "doc_gradient_descent", "text": "Gradient descent is an optimization algorithm used to minimize a loss function by updating parameters in the opposite direction of the gradient. The learning rate controls the size of each update."},
    {"source": "doc_neural_network", "text": "A neural network is composed of layers of interconnected neurons. Each connection has a weight, and training adjusts these weights using backpropagation and gradient descent."},
    {"source": "doc_rest_api", "text": "A REST API uses HTTP methods like GET, POST, and DELETE to let clients read and modify resources on a server. FastAPI is a Python framework that validates request data using type hints."},
    {"source": "doc_binary_search", "text": "Binary search finds an item in a sorted list by repeatedly dividing the search interval in half. It runs in O(log n) time, much faster than a linear scan for large lists."},
]

Writing knowledge_base.py


In [2]:
%%writefile assistant.py
"""
assistant.py
------------
The Day 20-style knowledge assistant being evaluated. Retrieval is real
(bag-of-words cosine similarity over knowledge_base.py, same approach as
every previous day). Generation is a scripted lookup table keyed by
question -- because to evaluate whether a judge catches real problems, the
assistant needs to actually produce some. 6 of the 20 answers below are
deliberately flawed: 2 with a wrong fact (correctness), 2 that omit part of
a multi-part question (completeness), and 2 that add a fabricated detail
not present in the retrieved context (groundedness). The other 14 are
straightforwardly correct. Which is which is NOT labeled here -- the whole
point of the evaluation framework in this notebook is to find them without
being told in advance.
"""

from __future__ import annotations

import math
import re
from collections import Counter
from typing import Dict, List

from knowledge_base import KNOWLEDGE_BASE


_STOPWORDS = {
    "a", "an", "the", "is", "are", "was", "were", "in", "on", "at", "by", "to", "of",
    "and", "or", "for", "with", "what", "who", "how", "why", "does", "do", "did",
    "it", "its", "this", "that", "as", "be", "been", "has", "have", "had",
}


def _embed(text: str) -> Counter:
    words = re.findall(r"[a-z0-9']+", text.lower())
    return Counter(w for w in words if w not in _STOPWORDS)


def _cosine_similarity(vec_a: Counter, vec_b: Counter) -> float:
    common = set(vec_a) & set(vec_b)
    dot = sum(vec_a[w] * vec_b[w] for w in common)
    mag_a = math.sqrt(sum(v * v for v in vec_a.values()))
    mag_b = math.sqrt(sum(v * v for v in vec_b.values()))
    return dot / (mag_a * mag_b) if mag_a and mag_b else 0.0


def retrieve_context(question: str) -> str:
    """
    Retrieve the single most relevant document for a question.

    Args:
        question: The user's question.

    Returns:
        The best-matching document's text (context for the judge).
    """
    q_vec = _embed(question)
    scored = [(_cosine_similarity(q_vec, _embed(d["text"])), d) for d in KNOWLEDGE_BASE]
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[0][1]["text"]


# Scripted answers, keyed by question text. See module docstring: 6 of
# these are deliberately flawed.
_ANSWERS: Dict[str, str] = {
    "Who created Python and when was it first released?":
        "Python was created by Guido van Rossum and released in 1989.",  # WRONG YEAR (correctness)
    "What kind of language is Python, and who is credited with creating it?":
        "Python is a programming language created by Guido van Rossum.",

    "How tall is the Eiffel Tower and when was it completed?":
        "The Eiffel Tower is 330 meters tall.",  # MISSING completion year (completeness)
    "What is the height of the Eiffel Tower in meters?":
        "The Eiffel Tower is 330 meters tall.",

    "How tall is Mount Everest?":
        "Mount Everest is 8849 meters tall.",
    "Is Mount Everest the tallest mountain on Earth?":
        "Yes, Mount Everest is the highest mountain on Earth, located in Nepal and China near the Khumbu Icefall.",  # FABRICATED location detail (groundedness)

    "Approximately how many square kilometers does the Amazon rainforest cover?":
        "The Amazon rainforest covers approximately 5500000 square kilometers.",
    "What geographic feature is the Amazon, and roughly how large is it?":
        "The Amazon is a rainforest covering approximately 5500000 square kilometers.",

    "When did the Apollo 11 moon landing occur?":
        "The Apollo 11 moon landing occurred on July 20, 1969.",
    "Who was the first person to walk on the Moon, and on what mission?":
        "Neil Armstrong was the first person to walk on the Moon.",  # MISSING mission name (completeness)

    "What is recursion in programming?":
        "Recursion is a programming technique where a function calls itself to solve smaller instances of a problem.",
    "Why does every recursive function need a base case?":
        "Because without a base case, the function would call itself forever, cause a stack overflow, and often crash the interpreter with a segmentation fault.",  # FABRICATED technical detail (groundedness)

    "What does gradient descent do?":
        "Gradient descent is an optimization algorithm that minimizes a loss function by updating parameters in the opposite direction of the gradient.",
    "What role does the learning rate play in gradient descent?":
        "The learning rate controls the size of each parameter update.",

    "What is a neural network made of?":
        "A neural network is composed of layers of interconnected neurons with weighted connections.",
    "How are a neural network's weights adjusted during training?":
        "A neural network's weights are adjusted through backpropagation and gradient descent.",

    "What HTTP methods does a REST API typically use?":
        "A REST API typically uses HTTP methods like GET and POST.",
    "How does FastAPI validate request data?":
        "FastAPI validates request data using XML schema definitions.",  # WRONG mechanism (correctness)

    "How does binary search find an item in a sorted list?":
        "Binary search finds an item by repeatedly dividing the search interval in half.",
    "What is the time complexity of binary search, and why is it faster than a linear scan?":
        "Binary search runs in O(log n) time because it halves the search space at each step rather than scanning every element.",
}


def answer_question(question: str) -> Dict[str, str]:
    """
    Answer a question using retrieval + (scripted) generation.

    Args:
        question: The user's question. Must be one of the 20 questions in
            eval_dataset.EVAL_DATASET (this is a scripted demo assistant,
            not a general-purpose one).

    Returns:
        {"answer": ..., "context": ...} -- the generated answer and the
        retrieved context it was (supposed to be) grounded in.

    Raises:
        KeyError: If the question isn't in the scripted answer set.
    """
    context = retrieve_context(question)
    answer = _ANSWERS[question]
    return {"answer": answer, "context": context}

Writing assistant.py


## 2. The 20-question labeled dataset

Two hand-written questions per source document, each with a manually written ground-truth
answer and a `key_points` list — the specific facts a *complete* answer needs to include,
used by the completeness dimension.


In [3]:
%%writefile eval_dataset.py
"""
eval_dataset.py
----------------
20 hand-labeled question/ground-truth pairs for the Day 20-style knowledge
assistant, 2 per document in the knowledge base. `key_points` is the set of
facts a *complete* answer must contain -- used by the completeness
dimension of the judge.
"""

from typing import Dict, List, TypedDict


class EvalItem(TypedDict):
    question: str
    ground_truth: str
    key_points: List[str]


EVAL_DATASET: List[EvalItem] = [
    {"question": "Who created Python and when was it first released?",
     "ground_truth": "Python was created by Guido van Rossum and first released in 1991.",
     "key_points": ["Guido van Rossum", "1991"]},
    {"question": "What kind of language is Python, and who is credited with creating it?",
     "ground_truth": "Python is a programming language created by Guido van Rossum.",
     "key_points": ["programming language", "Guido van Rossum"]},

    {"question": "How tall is the Eiffel Tower and when was it completed?",
     "ground_truth": "The Eiffel Tower is 330 meters tall and was completed in 1889.",
     "key_points": ["330 meters", "1889"]},
    {"question": "What is the height of the Eiffel Tower in meters?",
     "ground_truth": "The Eiffel Tower is 330 meters tall.",
     "key_points": ["330"]},

    {"question": "How tall is Mount Everest?",
     "ground_truth": "Mount Everest is 8849 meters tall.",
     "key_points": ["8849"]},
    {"question": "Is Mount Everest the tallest mountain on Earth?",
     "ground_truth": "Yes, Mount Everest is the highest mountain on Earth.",
     "key_points": ["highest mountain", "Earth"]},

    {"question": "Approximately how many square kilometers does the Amazon rainforest cover?",
     "ground_truth": "The Amazon rainforest covers approximately 5,500,000 square kilometers.",
     "key_points": ["5500000"]},
    {"question": "What geographic feature is the Amazon, and roughly how large is it?",
     "ground_truth": "The Amazon is a rainforest covering approximately 5,500,000 square kilometers.",
     "key_points": ["rainforest", "5500000"]},

    {"question": "When did the Apollo 11 moon landing occur?",
     "ground_truth": "The Apollo 11 moon landing occurred on July 20, 1969.",
     "key_points": ["July 20, 1969"]},
    {"question": "Who was the first person to walk on the Moon, and on what mission?",
     "ground_truth": "Neil Armstrong was the first person to walk on the Moon, on the Apollo 11 mission.",
     "key_points": ["Neil Armstrong", "Apollo 11"]},

    {"question": "What is recursion in programming?",
     "ground_truth": "Recursion is a programming technique where a function calls itself to solve smaller instances of a problem.",
     "key_points": ["function calls itself", "smaller instances"]},
    {"question": "Why does every recursive function need a base case?",
     "ground_truth": "Because without a base case, the function would call itself forever and cause a stack overflow.",
     "key_points": ["stack overflow"]},

    {"question": "What does gradient descent do?",
     "ground_truth": "Gradient descent is an optimization algorithm that minimizes a loss function by updating parameters in the opposite direction of the gradient.",
     "key_points": ["optimization algorithm", "minimizes a loss function"]},
    {"question": "What role does the learning rate play in gradient descent?",
     "ground_truth": "The learning rate controls the size of each parameter update.",
     "key_points": ["controls", "size"]},

    {"question": "What is a neural network made of?",
     "ground_truth": "A neural network is composed of layers of interconnected neurons with weighted connections.",
     "key_points": ["layers", "neurons"]},
    {"question": "How are a neural network's weights adjusted during training?",
     "ground_truth": "A neural network's weights are adjusted through backpropagation and gradient descent.",
     "key_points": ["backpropagation", "gradient descent"]},

    {"question": "What HTTP methods does a REST API typically use?",
     "ground_truth": "A REST API typically uses HTTP methods like GET, POST, and DELETE.",
     "key_points": ["GET", "POST"]},
    {"question": "How does FastAPI validate request data?",
     "ground_truth": "FastAPI validates request data using Python type hints.",
     "key_points": ["type hints"]},

    {"question": "How does binary search find an item in a sorted list?",
     "ground_truth": "Binary search finds an item by repeatedly dividing the search interval in half.",
     "key_points": ["dividing", "half"]},
    {"question": "What is the time complexity of binary search, and why is it faster than a linear scan?",
     "ground_truth": "Binary search runs in O(log n) time because it halves the search space at each step rather than scanning every element.",
     "key_points": ["O(log n)", "halves"]},
]

assert len(EVAL_DATASET) == 20

Writing eval_dataset.py


In [4]:
# Sanity check: retrieval finds the right document for every question in the eval set
from assistant import answer_question
from eval_dataset import EVAL_DATASET

for item in EVAL_DATASET[:3]:
    result = answer_question(item["question"])
    print(f"Q: {item['question']}")
    print(f"   context: {result['context'][:70]}...")
    print(f"   answer:  {result['answer']}")
    print()

Q: Who created Python and when was it first released?
   context: Python was first released by Guido van Rossum in 1991....
   answer:  Python was created by Guido van Rossum and released in 1989.

Q: What kind of language is Python, and who is credited with creating it?
   context: Python was first released by Guido van Rossum in 1991....
   answer:  Python is a programming language created by Guido van Rossum.

Q: How tall is the Eiffel Tower and when was it completed?
   context: The Eiffel Tower is 330 meters tall and was completed in 1889....
   answer:  The Eiffel Tower is 330 meters tall.



## 3. `llm_judge()`

Each dimension has a distinct, precise scoring rule (see the module docstring for the exact
criteria) rather than one vague "rate this 1-5" prompt:

- **Groundedness**: any number in the answer that doesn't appear in the context is a hard
  cap at 2/5 (a fabricated number is a hallucination, full stop). Otherwise scored by how much
  of the answer's vocabulary is traceable back to the context.
- **Correctness**: every number in the ground truth must appear in the answer; a missing or
  substituted number caps the score at 2/5.
- **Completeness**: the fraction of hand-labeled `key_points` actually present in the answer.

Building this heuristic surfaced two real bugs worth calling out (both fixed here, both
discussed again in the limitations section): a stopword-blind retriever that briefly matched
questions to the wrong document on words like "is" and "in", and a number-extraction regex
that treated a sentence-ending period after a number (`"...1969."`) as a decimal point,
producing false "hallucinated number" flags.


In [5]:
%%writefile judge.py
"""
judge.py
--------
llm_judge(question, context, answer, ground_truth) scores an answer on three
dimensions, 1-5 each, and returns a parsed dict -- exactly the shape a real
GPT-4o-mini structured-output call would return. As with every previous
day's mock, this runs on a deterministic offline heuristic scorer by
default; set OPENAI_API_KEY to swap in a real judge call with no change to
how the rest of the notebook calls llm_judge().

THE THREE DIMENSIONS, with precise criteria (as the task requires):

- Groundedness (1-5): the answer is supported ONLY by the retrieved
  context, with no fabricated additions. Scored by checking whether every
  number in the answer also appears in the context, and how much of the
  answer's non-trivial vocabulary is traceable to the context. An answer
  that introduces a number or a specific named claim absent from the
  context is a hallucination, and is capped at 2 regardless of everything
  else looking fine.
- Correctness (1-5): the answer matches the known ground truth. Scored by
  checking whether every number in the ground truth also appears in the
  answer, AND whether the answer doesn't contain a *different* number
  where the ground truth's number should be (catching "confidently wrong"
  answers, not just incomplete ones).
- Completeness (1-5): the full question is answered with nothing important
  omitted. Scored against the hand-labeled `key_points` for that question
  -- the fraction of key points actually present in the answer.
"""

from __future__ import annotations

import re
from typing import Dict, List


def _numbers_in(text: str) -> List[str]:
    # (?:\.\d+)? requires digits AFTER a decimal point to count it -- without
    # this, a sentence-ending period right after a number (e.g. "...1969.")
    # gets swallowed into the match as if it were a decimal point, making
    # "1969." != "1969" and producing a false "unsupported number" flag.
    return re.findall(r"\d[\d,]*(?:\.\d+)?", text.replace(",", ""))


def _contains_phrase(answer: str, phrase: str) -> bool:
    return phrase.lower() in answer.lower()


def _score_groundedness(context: str, answer: str) -> int:
    context_numbers = set(_numbers_in(context))
    answer_numbers = set(_numbers_in(answer))
    unsupported_numbers = answer_numbers - context_numbers
    if unsupported_numbers:
        return 2  # a fabricated number is a hard cap, regardless of everything else

    context_words = set(re.findall(r"[a-z']+", context.lower()))
    answer_words = [w for w in re.findall(r"[a-z']+", answer.lower()) if len(w) > 3]
    if not answer_words:
        return 3
    traceable = sum(1 for w in answer_words if w in context_words)
    ratio = traceable / len(answer_words)

    if ratio >= 0.75:
        return 5
    if ratio >= 0.5:
        return 4
    if ratio >= 0.3:
        return 3
    return 2


def _score_correctness(ground_truth: str, answer: str) -> int:
    gt_numbers = _numbers_in(ground_truth)
    answer_numbers = set(_numbers_in(answer))

    if gt_numbers:
        missing = [n for n in gt_numbers if n not in answer_numbers]
        # a *wrong* number present where the ground truth's number is absent
        # is worse than simply omitting it -- still caught as "missing" here,
        # but combined with groundedness's fabricated-number check, a wrong
        # number gets penalized on both dimensions, which is the correct
        # outcome: it's both incorrect AND unsupported by context.
        if not missing:
            return 5
        return 2  # got a number-bearing fact wrong or omitted it entirely

    # No numeric fact to check -- fall back to key-phrase overlap.
    gt_phrases = [p.strip() for p in re.split(r",| and ", ground_truth) if len(p.strip()) > 8]
    if not gt_phrases:
        return 4
    hits = sum(1 for p in gt_phrases if _contains_phrase(answer, p[:20]))
    ratio = hits / len(gt_phrases)
    if ratio >= 0.8:
        return 5
    if ratio >= 0.5:
        return 4
    return 3


def _score_completeness(key_points: List[str], answer: str) -> int:
    if not key_points:
        return 5
    present = sum(1 for kp in key_points if _contains_phrase(answer, kp))
    ratio = present / len(key_points)
    if ratio == 1.0:
        return 5
    if ratio >= 0.5:
        return 3
    return 1


def llm_judge(question: str, context: str, answer: str, ground_truth: str,
              key_points: List[str] = None) -> Dict[str, int]:
    """
    Score an assistant's answer on groundedness, correctness, and completeness.

    Args:
        question: The original question (kept in the signature to match a
            real judge prompt, which would include it for context, even
            though this heuristic scorer doesn't need it directly).
        context: The retrieved context the answer should be grounded in.
        answer: The assistant's answer being evaluated.
        ground_truth: The hand-written correct answer.
        key_points: Facts a complete answer must include, for the
            completeness dimension. Defaults to an empty list.

    Returns:
        {"groundedness": 1-5, "correctness": 1-5, "completeness": 1-5}
    """
    key_points = key_points or []
    return {
        "groundedness": _score_groundedness(context, answer),
        "correctness": _score_correctness(ground_truth, answer),
        "completeness": _score_completeness(key_points, answer),
    }

Writing judge.py


In [6]:
# Run the judge over all 20 questions and inspect every score
from judge import llm_judge

results = {}
for item in EVAL_DATASET:
    r = answer_question(item["question"])
    scores = llm_judge(item["question"], r["context"], r["answer"], item["ground_truth"], item["key_points"])
    results[item["question"]] = {"answer": r["answer"], "context": r["context"], "scores": scores}

print(f"{'Question':<58} | {'G':>2} {'C':>2} {'Comp':>4}")
print("-" * 72)
for question, data in results.items():
    s = data["scores"]
    flag = "  <-- lowest-scoring" if min(s.values()) <= 2 else ""
    print(f"{question[:58]:<58} | {s['groundedness']:>2} {s['correctness']:>2} {s['completeness']:>4}{flag}")

Question                                                   |  G  C Comp
------------------------------------------------------------------------
Who created Python and when was it first released?         |  2  2    3  <-- lowest-scoring
What kind of language is Python, and who is credited with  |  4  5    5
How tall is the Eiffel Tower and when was it completed?    |  5  2    3  <-- lowest-scoring
What is the height of the Eiffel Tower in meters?          |  5  5    5
How tall is Mount Everest?                                 |  5  5    5
Is Mount Everest the tallest mountain on Earth?            |  3  5    5
Approximately how many square kilometers does the Amazon r |  5  5    5
What geographic feature is the Amazon, and roughly how lar |  5  5    5
When did the Apollo 11 moon landing occur?                 |  5  5    5
Who was the first person to walk on the Moon, and on what  |  5  2    3  <-- lowest-scoring
What is recursion in programming?                          |  5  5    5
Why

## 4. Aggregate scores and the lowest dimension

Averaging each dimension across all 20 questions, and identifying which one is weakest.


In [7]:
dims = ["groundedness", "correctness", "completeness"]
averages = {d: round(sum(v["scores"][d] for v in results.values()) / len(results), 3) for d in dims}

print("Aggregate scores (out of 5):")
for d, avg in averages.items():
    print(f"  {d:<14}: {avg}")

lowest_dim = min(averages, key=averages.get)
gap = min(v for k, v in averages.items() if k != lowest_dim) - averages[lowest_dim]
print()
print(f"Lowest dimension: {lowest_dim} ({averages[lowest_dim]}), "
      f"{gap:.3f} below the next-lowest dimension.")

Aggregate scores (out of 5):
  groundedness  : 4.35
  correctness   : 4.55
  completeness  : 4.5

Lowest dimension: groundedness (4.35), 0.150 below the next-lowest dimension.


**Groundedness comes out lowest**, and looking at *why* is more informative than the
number itself: the two numeric hallucinations (wrong year, wrong height) hit the hard cap of
2/5 immediately, but the two fabricated-*descriptive*-detail cases (the invented mountain
location, the invented "segmentation fault") only scored 3/5 — noticeably below a clean
answer's 4-5, but nowhere near as harshly penalized as the numeric hallucinations. That gap
between how the judge treats a fabricated number versus a fabricated phrase is itself a real,
worth-documenting limitation — see section 7.


## 5. `regression_test_runner()`

Saves the current aggregate scores as a baseline, then re-runs the full 20-question suite and
compares every dimension against it — printing PASS or FAIL per dimension, with the exact
score drop named for anything that breaches the threshold. This is the check that should run
automatically before any deployment, the way a CI test suite blocks a bad merge.


In [8]:
%%writefile regression.py
"""
regression.py
---------------
regression_test_runner() re-runs the full 20-question eval suite, compares
each dimension's average score against a stored baseline, and prints
PASS/FAIL per dimension -- exactly the kind of check that should block a
deployment if a code change quietly made the assistant worse.
"""

from __future__ import annotations

import json
from pathlib import Path
from typing import Callable, Dict, List

from eval_dataset import EVAL_DATASET
from judge import llm_judge

BASELINE_PATH = Path("baseline_scores.json")
REGRESSION_THRESHOLD = 0.15  # a dimension average dropping by more than this fails the run


def run_full_evaluation(answer_fn: Callable[[str], Dict[str, str]]) -> Dict[str, float]:
    """
    Run the 20-question suite against a given answer function and return
    aggregate (average) scores per dimension.

    Args:
        answer_fn: A function question -> {"answer": ..., "context": ...},
            e.g. assistant.answer_question. Taking this as a parameter
            (rather than hardcoding assistant.answer_question) is what
            lets the same runner evaluate a "before" and an "after"
            version of the assistant for the regression demo below.

    Returns:
        {"groundedness": avg, "correctness": avg, "completeness": avg}
    """
    totals = {"groundedness": 0.0, "correctness": 0.0, "completeness": 0.0}
    for item in EVAL_DATASET:
        result = answer_fn(item["question"])
        scores = llm_judge(
            item["question"], result["context"], result["answer"],
            item["ground_truth"], item["key_points"],
        )
        for dim in totals:
            totals[dim] += scores[dim]

    n = len(EVAL_DATASET)
    return {dim: round(total / n, 3) for dim, total in totals.items()}


def save_baseline(scores: Dict[str, float]) -> None:
    """Write the current aggregate scores to disk as the regression baseline."""
    BASELINE_PATH.write_text(json.dumps(scores, indent=2))


def load_baseline() -> Dict[str, float]:
    """Load the stored baseline scores. Raises FileNotFoundError if none exists yet."""
    if not BASELINE_PATH.exists():
        raise FileNotFoundError(
            f"No baseline found at {BASELINE_PATH}. Run save_baseline() once before "
            "calling regression_test_runner()."
        )
    return json.loads(BASELINE_PATH.read_text())


def regression_test_runner(answer_fn: Callable[[str], Dict[str, str]]) -> bool:
    """
    Run the full 20-question suite against answer_fn and compare each
    dimension's average against the stored baseline.

    Args:
        answer_fn: The assistant version to test, as question -> {"answer", "context"}.

    Returns:
        True if every dimension is within REGRESSION_THRESHOLD of the
        baseline (overall PASS), False if any dimension dropped more than
        that (overall FAIL). Also prints a PASS/FAIL line per dimension,
        naming the specific score drop for any dimension that breached the
        threshold.
    """
    baseline = load_baseline()
    current = run_full_evaluation(answer_fn)

    overall_pass = True
    print(f"{'Dimension':<14} | {'Baseline':>8} | {'Current':>8} | {'Delta':>7} | Result")
    print("-" * 60)
    for dim in ("groundedness", "correctness", "completeness"):
        delta = current[dim] - baseline[dim]
        breached = delta < -REGRESSION_THRESHOLD
        if breached:
            overall_pass = False
        status = "FAIL" if breached else "PASS"
        print(f"{dim:<14} | {baseline[dim]:>8.3f} | {current[dim]:>8.3f} | {delta:>+7.3f} | {status}")
        if breached:
            print(f"  -> {dim} dropped by {abs(delta):.3f} (threshold: {REGRESSION_THRESHOLD}). "
                  f"This would block a deployment.")

    print("-" * 60)
    print(f"Overall: {'PASS' if overall_pass else 'FAIL'}")
    return overall_pass

Writing regression.py


In [9]:
from regression import run_full_evaluation, save_baseline, regression_test_runner

baseline_scores = run_full_evaluation(answer_question)
print("Baseline scores:", baseline_scores)
save_baseline(baseline_scores)

Baseline scores: {'groundedness': 4.35, 'correctness': 4.55, 'completeness': 4.5}


## 6. Proving the regression test actually catches a regression

Two runs: the same assistant against its own baseline (should PASS cleanly), and a
deliberately regressed version — several *additional* corrupted answers, simulating a bad
code change — against the same baseline (should FAIL, with the specific dimensions and score
drops named).


In [10]:
%%writefile assistant_broken.py
"""
assistant_broken.py
---------------------
A stand-in for "someone shipped a bad change" -- reuses the same retrieval
as assistant.py, but several more answers have been corrupted (extra
fabricated numbers, wrong facts, dropped details) on top of the original
6 flaws. Used only to prove regression_test_runner() actually catches a
real drop, not just to re-confirm the baseline passes against itself.
"""

from __future__ import annotations

from typing import Dict

from assistant import retrieve_context

_BROKEN_ANSWERS: Dict[str, str] = {
    "Who created Python and when was it first released?":
        "Python was created by Guido van Rossum and released in 1989.",  # (unchanged flaw)
    "What kind of language is Python, and who is credited with creating it?":
        "Python is a programming language, version 3.12, created by Guido van Rossum.",  # NEW fabricated version number
    "How tall is the Eiffel Tower and when was it completed?":
        "The Eiffel Tower is 330 meters tall.",  # (unchanged flaw)
    "What is the height of the Eiffel Tower in meters?":
        "The Eiffel Tower is approximately 324 meters tall.",  # NEW wrong number
    "How tall is Mount Everest?":
        "Mount Everest is 8849 meters tall.",
    "Is Mount Everest the tallest mountain on Earth?":
        "Yes, Mount Everest is the highest mountain on Earth, located in Nepal and China near the Khumbu Icefall.",
    "Approximately how many square kilometers does the Amazon rainforest cover?":
        "The Amazon rainforest covers approximately 4200000 square kilometers.",  # NEW wrong number
    "What geographic feature is the Amazon, and roughly how large is it?":
        "The Amazon is a rainforest.",  # NEW dropped the size entirely
    "When did the Apollo 11 moon landing occur?":
        "The Apollo 11 moon landing occurred on July 20, 1969.",
    "Who was the first person to walk on the Moon, and on what mission?":
        "Neil Armstrong was the first person to walk on the Moon.",
    "What is recursion in programming?":
        "Recursion is a programming technique where a function calls itself to solve smaller instances of a problem.",
    "Why does every recursive function need a base case?":
        "Because without a base case, the function would call itself forever, cause a stack overflow, and often crash the interpreter with a segmentation fault.",
    "What does gradient descent do?":
        "Gradient descent is an optimization algorithm that minimizes a loss function by updating parameters in the opposite direction of the gradient.",
    "What role does the learning rate play in gradient descent?":
        "The learning rate has no significant effect on training.",  # NEW factually wrong
    "What is a neural network made of?":
        "A neural network is composed of layers of interconnected neurons with weighted connections.",
    "How are a neural network's weights adjusted during training?":
        "A neural network's weights are adjusted through backpropagation and gradient descent.",
    "What HTTP methods does a REST API typically use?":
        "A REST API typically uses HTTP methods like GET and POST.",
    "How does FastAPI validate request data?":
        "FastAPI validates request data using XML schema definitions.",
    "How does binary search find an item in a sorted list?":
        "Binary search finds an item by repeatedly dividing the search interval in half.",
    "What is the time complexity of binary search, and why is it faster than a linear scan?":
        "Binary search runs in O(n) time because it checks fewer elements than a linear scan.",  # NEW wrong complexity
}


def answer_question(question: str) -> Dict[str, str]:
    """Same interface as assistant.answer_question(), but using the corrupted answer set."""
    context = retrieve_context(question)
    answer = _BROKEN_ANSWERS[question]
    return {"answer": answer, "context": context}

Writing assistant_broken.py


In [11]:
from assistant_broken import answer_question as broken_answer_question

print("=== Regression test: unchanged assistant (expected: PASS) ===")
regression_test_runner(answer_question)
print()
print("=== Regression test: deliberately regressed assistant (expected: FAIL) ===")
regression_test_runner(broken_answer_question)

=== Regression test: unchanged assistant (expected: PASS) ===
Dimension      | Baseline |  Current |   Delta | Result
------------------------------------------------------------
groundedness   |    4.350 |    4.350 |  +0.000 | PASS
correctness    |    4.550 |    4.550 |  +0.000 | PASS
completeness   |    4.500 |    4.500 |  +0.000 | PASS
------------------------------------------------------------
Overall: PASS

=== Regression test: deliberately regressed assistant (expected: FAIL) ===
Dimension      | Baseline |  Current |   Delta | Result
------------------------------------------------------------
groundedness   |    4.350 |    3.900 |  -0.450 | FAIL
  -> groundedness dropped by 0.450 (threshold: 0.15). This would block a deployment.
correctness    |    4.550 |    4.000 |  -0.550 | FAIL
  -> correctness dropped by 0.550 (threshold: 0.15). This would block a deployment.
completeness   |    4.500 |    3.600 |  -0.900 | FAIL
  -> completeness dropped by 0.900 (threshold: 0.15). This w

False

## 7. Limitations of LLM-as-judge evaluation, honestly

Grounded in what actually happened while building this judge, not a generic disclaimer list:

**1. A heuristic (or an LLM) judge scoring on lexical/semantic overlap will penalize correct
paraphrasing it can't verify.** The clean answer *"Python is a programming language created by
Guido van Rossum"* initially scored groundedness 2/5, because "programming language" doesn't
appear verbatim in a context that only says "Python was first released by...". That's not a
hallucination — it's common knowledge stated in different words — but a judge that only checks
lexical overlap can't tell the difference between a reasonable paraphrase and an invented
claim. Loosening the scoring thresholds fixed this specific case, but the underlying problem
doesn't fully go away: **any lexical-overlap judge will have some false-positive rate on
legitimate rephrasing**, and the only real fix is a judge that understands semantic
entailment, not just word overlap — which is exactly why real LLM-as-judge setups use an LLM
here instead of a heuristic, and why even *those* aren't immune to the same failure mode in
harder cases.

**2. The judge scores different *kinds* of fabrication very differently, and not always for a
good reason.** A fabricated *number* (wrong year, wrong height) got caught hard here — capped
at groundedness 2/5 immediately, because numbers are easy to check against the context
mechanically. A fabricated *descriptive claim* (an invented mountain location, an invented
technical detail) only dropped the score to 3/5 — genuinely lower than a clean answer, but far
more forgiving than the numeric case, purely because "checking whether this specific claim is
supported" is much harder than "checking whether this number appears in the text." **This
means a judge like this one is systematically better at catching fabricated facts and figures
than fabricated qualitative claims** — worth knowing before trusting a groundedness score to
catch every kind of hallucination equally.

**3. Two of this judge's early bugs were regex bugs, not reasoning bugs — but they'd have
produced the same wrong verdict either way.** The retriever initially matched "What is
recursion in programming?" to the Eiffel Tower document because stopwords like "is" and "in"
dominated the bag-of-words similarity score. The number extractor initially flagged "1969."
(with a trailing sentence period) as an unsupported number, because "1969" ≠ "1969." as
strings. Neither bug was a judgment call gone wrong — they were plumbing bugs upstream of any
actual scoring logic. **This is a limitation of testing an evaluation system at all: a bug in
the harness looks identical to a bug in the judgment from the outside** (a wrong score is a
wrong score), so an evaluation framework needs its own tests/spot-checks on obviously-correct
cases, not just trust in the scoring rubric being sound in principle.

**4. When human evaluation is still necessary.** This judge can check "does this number match"
and "does this phrase appear," which covers factual QA well. It has no way to judge tone,
whether an explanation is pedagogically clear versus just technically accurate, whether a
summary preserved the *emphasis* of the source material and not just its facts, or whether an
answer to an open-ended or subjective question is "good" at all — none of those have a
ground-truth string to check against. Any of those question types still need a human in the
loop; an automated judge (heuristic or LLM) is a regression-test safety net for the kinds of
errors it can mechanically check, not a replacement for human review of answer quality more
broadly.
